# 💧 Water Quality Classifier

## 0. Environment setup

Imports, device detection, and helpful utility functions. Ensure required libraries are installed.


In [ ]:
import os, sys
from pathlib import Path
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from sklearn.metrics import classification_report, confusion_matrix

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

BASE_DIR = Path('.').resolve()
DATA_DIR = BASE_DIR / 'data' / 'water_dataset'
print('Data dir:', DATA_DIR)


## 1. Data preprocessing

Verify images (remove corrupted), define transforms, and create datasets/loaders.


In [ ]:
remove_corrupted = True  # set to False to only report corrupted files

for main_folder in ['train', 'test']:
    for folder in ['clean', 'muddy', 'polluted']:
        path = DATA_DIR / main_folder / folder
        if not path.exists():
            print('Warning: path not found:', path)
            continue
        for file in os.listdir(path):
            fpath = path / file
            try:
                Image.open(fpath).verify()
            except Exception as e:
                print('Corrupted:', fpath, '->', e)
                if remove_corrupted:
                    try:
                        os.remove(fpath)
                        print('Removed corrupted file:', fpath)
                    except Exception as e2:
                        print('Failed to remove:', fpath, e2)


In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    transforms.RandomResizedCrop(128, scale=(0.7, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)
])

test_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)
])

train_dataset = None
if (DATA_DIR / 'train').exists():
    train_dataset = datasets.ImageFolder(root=str(DATA_DIR / 'train'), transform=train_transform)
else:
    print('Train folder not found')

if (DATA_DIR / 'test').exists():
    test_dataset = datasets.ImageFolder(root=str(DATA_DIR / 'test'), transform=test_transform)
else:
    test_dataset = None

if train_dataset is not None:
    val_size = int(0.2 * len(train_dataset))
    train_size = len(train_dataset) - val_size
    train_ds, val_ds = random_split(train_dataset, [train_size, val_size])
    train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)
else:
    train_loader, val_loader = None, None

test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False) if test_dataset is not None else None

if train_dataset is not None:
    print('Classes:', train_dataset.classes)


## 2. Model definition

WaterCNN architecture.


In [ ]:
class WaterCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Dropout(0.4),
            nn.Linear(128, 3)
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

model = WaterCNN().to(device)
print('Model ready')


## 3. Training 

In [ ]:
run_training = False

if run_training:
    if train_loader is None or val_loader is None:
        raise RuntimeError('Train/val loaders not available')
    model = WaterCNN().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, verbose=True)

    best_val_loss = float('inf')
    patience, trigger_times = 5, 0
    num_epochs = 2

    from tqdm.notebook import tqdm
    for epoch in range(num_epochs):
        model.train()
        total_loss, correct, total = 0, 0, 0
        for imgs, labels in tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}'):
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        avg_train_loss = total_loss / len(train_loader)
        train_acc = 100 * correct / total

        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                outputs = model(imgs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                _, preds = torch.max(outputs, 1)
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)

        avg_val_loss = val_loss / len(val_loader)
        val_acc = 100 * val_correct / val_total
        print(f'Epoch {epoch+1}: Train Loss: {avg_train_loss:.4f}, Acc: {train_acc:.2f}% | Val Loss: {avg_val_loss:.4f}, Acc: {val_acc:.2f}%')
        scheduler.step(avg_val_loss)

        MODEL_PATH = BASE_DIR / 'week2_Model_Training' / 'water_quality_classifier.pth'
        MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
        if avg_val_loss < best_val_loss - 1e-4:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), MODEL_PATH)
            print('Model improved. Saved :', MODEL_PATH)
            trigger_times = 0
        else:
            trigger_times += 1
            if trigger_times >= patience:
                print('Early stopping at epoch', epoch+1)
                break

    print('Training finished')
else:
    print('Training skipped. Set run_training=True to run it here')


## 4. Evaluation

Evaluate saved model on test set and save logs.


In [ ]:
MODEL_PATH = BASE_DIR / 'week2_Model_Training' / 'water_quality_classifier.pth'
OUTPUTS_DIR = BASE_DIR / 'outputs'
LOGS_DIR = OUTPUTS_DIR / 'logs'
LOGS_DIR.mkdir(parents=True, exist_ok=True)

if not MODEL_PATH.exists():
    print('Model not found at', MODEL_PATH)
else:
    model = WaterCNN().to(device)
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    model.eval()

    if test_loader is None:
        print('test_loader not available')
    else:
        y_true, y_pred = [], []
        with torch.no_grad():
            for imgs, labels in test_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                outputs = model(imgs)
                _, preds = torch.max(outputs, 1)
                y_true.extend(labels.cpu().numpy())
                y_pred.extend(preds.cpu().numpy())

        acc = 100 * np.mean(np.array(y_true) == np.array(y_pred))
        print(f'Test Accuracy: {acc:.2f}%')

        report = classification_report(y_true, y_pred, target_names=(train_dataset.classes if train_dataset is not None else None))
        conf_matrix = confusion_matrix(y_true, y_pred)
        timestamp = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
        log_file = LOGS_DIR / f'evaluation_log_{timestamp}.txt'

        with open(log_file, 'w') as f:
            f.write(f'Test Accuracy: {acc:.2f}%\n\n')
            f.write('Classification Report:\n')
            f.write(report)
            f.write('\nConfusion Matrix:\n')
            import numpy as _np
            _np.savetxt(f, conf_matrix, fmt='%d')
        print('Evaluation log saved to:', log_file)


## 5. Visualization

Save sample predicted images and example graphs to outputs/.


In [ ]:
PRED_DIR = BASE_DIR / 'outputs' / 'predictions'
GRAPHS_DIR = BASE_DIR / 'outputs' / 'graphs'
PRED_DIR.mkdir(parents=True, exist_ok=True)
GRAPHS_DIR.mkdir(parents=True, exist_ok=True)

if not MODEL_PATH.exists():
    print('Model not found -> skip visualization')
else:
    model = WaterCNN().to(device)
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    model.eval()

    if test_dataset is None:
        print('test_dataset not available')
    else:
        for i in range(min(10, len(test_dataset))):
            image, true_label = test_dataset[i]
            with torch.no_grad():
                output = model(image.unsqueeze(0).to(device))
                _, predicted = torch.max(output, 1)
                label = predicted.item()
                img_vis = image * 0.5 + 0.5
                import matplotlib.pyplot as _plt
                _plt.imshow(img_vis.permute(1,2,0))
                _plt.title(f'Predicted: {label}, Actual: {true_label}')
                _plt.axis('off')
                _plt.savefig(PRED_DIR / f'prediction_{i+1}.png')
                _plt.close()
        print('Saved predictions to', PRED_DIR)

        import matplotlib.pyplot as _plt2
        _plt2.figure(figsize=(5,4))
        _plt2.bar(['Accuracy'], [90])
        _plt2.ylim(0,100)
        _plt2.title('Example Accuracy')
        timestamp = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
        _plt2.savefig(GRAPHS_DIR / f'accuracy_plot_{timestamp}.png')
        _plt2.close()
        print('Saved graphs to', GRAPHS_DIR)
